# 02 — Baseline com Ridge

Modelo linear regularizado como baseline. Usa o **mesmo split** que será aplicado
no grid search principal (`split_holdout(random_state=42)`), para que a comparação
RF/XGB × baseline seja apples-to-apples.

O objetivo é fixar um piso: qualquer modelo mais complexo precisa bater este número
para justificar custo computacional.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data import load_data, split_holdout
from src.preprocessing import prepare_features

In [2]:
treino, _ = load_data()
treino_dev, treino_holdout = split_holdout(treino, test_size=0.2, random_state=42)
X_dev, y_dev = prepare_features(treino_dev)
X_h, y_h = prepare_features(treino_holdout)
print('dev:', X_dev.shape, '| holdout:', X_h.shape)

dev: (1168, 48) | holdout: (292, 48)


## Treina Ridge com alpha padrão (1.0)

Ridge é sensível à escala — mas como vamos usá-lo só como baseline e o grid_search
do projeto principal usa árvores (insensíveis à escala), aceitamos a simplicidade aqui.

In [3]:
model = Ridge(alpha=1.0, random_state=42)
model.fit(X_dev, y_dev)

pred_log = model.predict(X_h)
pred = np.expm1(pred_log)
actual = np.expm1(y_h).values

rmsle = float(np.sqrt(mean_squared_error(y_h, pred_log)))
mae = float(mean_absolute_error(actual, pred))
r2 = float(r2_score(actual, pred))

print(f'Ridge baseline — holdout (random_state=42)')
print(f'  RMSLE:  {rmsle:.4f}')
print(f'  MAE:    US$ {mae:,.0f}')
print(f'  R²:     {r2:.4f}')

Ridge baseline — holdout (random_state=42)
  RMSLE:  0.1395
  MAE:    US$ 18,595
  R²:     0.8847


## Conclusão

RandomForest e XGBoost precisam **bater o RMSLE acima** para se justificarem.
Compare com `docs/metrics.json` — o modelo vencedor (XGB) está em ~0.139, o que
representa redução vs. baseline.